In [4]:
# 2. 필요한 모듈 임포트
import pandas as pd
import plotly.io as pio
from IPython.display import display

from services.data_preparer import prepare_proposal_01_data
from services.proposal_views.proposal_01_view import create_figure_and_df

# --- 테스트 실행 ---

# 1. app.py의 역할 (1): 데이터 준비 함수를 호출하여 필요한 데이터를 미리 로드합니다.
print("Step 1: `data_preparer`를 통해 분석용 데이터와 순서 정보를 준비합니다...")
data_bundle = prepare_proposal_01_data() # 글로벌 필터는 모두 기본값('전체')으로 호출
analysis_df = data_bundle.get("analysis_df", pd.DataFrame())
order_map = data_bundle.get("order_map", {})
print(" -> 데이터 준비 완료!")


# 2. app.py의 역할 (2): 사용자가 Streamlit 위젯을 통해 필터를 선택했다고 가정합니다.
# 이 값들을 바꾸면서 여러 경우를 테스트해볼 수 있습니다.

# 2-1. app.py에 있을 차원 설정(Dimension Config) 정의
DIMENSION_CONFIG = {
    '부서별': {'type': 'hierarchical', 'top': 'DIVISION_NAME', 'sub': 'OFFICE_NAME'},
    '직무별': {'type': 'hierarchical', 'top': 'JOB_L1_NAME', 'sub': 'JOB_L2_NAME'},
    '성별': {'type': 'flat', 'col': 'GENDER', 'order': ['남성', '여성']},
    '연령별': {'type': 'flat', 'col': 'AGE_BIN'},
    '경력연차별': {'type': 'flat', 'col': 'CAREER_BIN'},
    '연봉구간별': {'type': 'flat', 'col': 'SALARY_BIN'},
    '지역별': {'type': 'flat', 'col': 'REGION_CATEGORY'},
    '계약별': {'type': 'flat', 'col': 'CONT_CATEGORY'}
}

# 2-2. 사용자 선택 시뮬레이션
# ----- 테스트하고 싶은 값으로 변경 -----
selected_dimension_ui = '경력연차별' # 예: '부서별', '직무별', '성별'
drilldown_selection = '전체'   # '부서별' 선택 시 'Development Division' 등으로 변경 가능
# -----------------------------------

print(f"Step 2: 사용자가 '{selected_dimension_ui}' 차원을, '{drilldown_selection}' 그룹으로 보기를 선택했습니다.")


# 3. app.py의 역할 (3): view 함수에 준비된 모든 데이터와 필터 값을 전달하여 결과물 생성
print("Step 3: `view` 모듈을 호출하여 그래프와 요약 테이블을 생성합니다...")
if not analysis_df.empty:
    # dimension_config에 order 정보 추가 (view 함수가 사용할 수 있도록)
    for k, v in DIMENSION_CONFIG.items():
        if v['type'] == 'hierarchical':
            v['order'] = order_map.get(v['top'])
            v['sub_order'] = order_map.get(v['sub'])
        else: # flat
            v['order'] = order_map.get(v['col'])

    fig, aggregate_df = create_figure_and_df(
        analysis_df=analysis_df, 
        dimension_ui_name=selected_dimension_ui, 
        drilldown_selection=drilldown_selection,
        dimension_config=DIMENSION_CONFIG,
        order_map=order_map # order_map 전달
    )
    print(" -> 생성 완료!")
else:
    print(" -> 분석할 데이터가 없어 빈 결과물을 생성합니다.")
    fig, aggregate_df = go.Figure(), pd.DataFrame()


# --- 결과 확인 ---

# 4. ipynb에서 생성된 그래프를 확인합니다.
print("\n--- [결과 1] 생성된 Plotly 그래프 ---")
# pio.renderers.default = 'vscode' 
fig.show()

# 5. ipynb에서 생성된 요약 테이블(aggregate_df)을 확인합니다.
print(f"\n--- [결과 2] '{selected_dimension_ui}' 기준 생성된 요약 테이블 ---")
display(aggregate_df)

Step 1: `data_preparer`를 통해 분석용 데이터와 순서 정보를 준비합니다...
 -> 데이터 준비 완료!
Step 2: 사용자가 '경력연차별' 차원을, '전체' 그룹으로 보기를 선택했습니다.
Step 3: `view` 모듈을 호출하여 그래프와 요약 테이블을 생성합니다...
 -> 생성 완료!

--- [결과 1] 생성된 Plotly 그래프 ---



--- [결과 2] '경력연차별' 기준 생성된 요약 테이블 ---


CAREER_BIN,전체 평균,1~3년,3~7년,7~15년,15년 이상
PROMOTION_STEP,,,,,
Staff → Manager,5.76,2.4,3.81,6.27,5.40
Manager → Director,6.12,-,3.93,6.15,7.45
